# Agile Deliberation Demo

Licensed under the Apache License, Version 2.0.

In [ ]:
# @title Dependencies

%load_ext autoreload
%autoreload 2

import numpy as np
import tensorflow as tf

import logging
import os
import pickle
import slugify
import warnings

from IPython.display import HTML, display

from agile_deliberation_lib import utils as utils
from agile_deliberation_lib import image as image_py
from agile_deliberation_lib import definitions as definitions_py
from agile_deliberation_lib import annotator as annotator_py
from agile_deliberation_lib import llm as llm_py
from agile_deliberation_lib import components as components_py
from agile_deliberation_lib import classifier as classifier_py
from agile_deliberation_lib import search_images as search_images_py
from agile_deliberation_lib import refine_definition as refine_definition_py
from agile_deliberation_lib import reflection as reflection_py
from agile_deliberation_lib import interaction as interaction_py
from agile_deliberation_lib import diverse_sample as diverse_sample_py
from agile_deliberation_lib import deliberators as deliberators_py

from agile_deliberation_lib.retrieval import RetrievalClient

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings('ignore',category=UserWarning, module='google.protobuf.runtime_version')


# automatically wrap long lines of text in the output
def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))

get_ipython().events.register('pre_run_cell', set_css)


Definition = definitions_py.Definition
MyImage = image_py.MyImage


logging.basicConfig(
    encoding='utf-8',
    level=logging.INFO,
    force=True,
    format='%(levelname)s - %(funcName)s - %(lineno)d: %(message)s'
)

concept_deliberator = None

In [ ]:
#@title Setup the retrieval client using nearest neighbor.

# Point to the indices path specified in the indices_train.json file,
# which can be found in the code repository.
retrieval_client = RetrievalClient(indices_path="indices_train.json")

In [ ]:
# @title Setup the LLM and test if it works.

# @markdown Provide an API key or set the GEMINI_API_KEY environment variable.
api_key = ''  # @param {type:"string"}

# @markdown Customize the model names if needed.
expensive_model_name = 'gemini-2.5-pro'  # @param {type:"string"}
cheap_model_name = 'gemini-2.5-flash'  # @param {type:"string"}

model_client = llm_py.ModelClient(
    client_names=[cheap_model_name] if cheap_model_name else None,
    api_key=api_key if api_key else None,
    client_name=expensive_model_name if expensive_model_name else llm_py.DEFAULT_MODEL_NAME,
)

NUM_IMAGES = 3

images = retrieval_client.text_image_search('pandas', NUM_IMAGES)

for i in range(min(NUM_IMAGES, len(images))):
  print(model_client.gemini_image_prompt('what is in this image, answer in one sentence', [images[i]]))

# ***Experiment Tasks***

# Instructions

Imagine you are a moderator for a Reddit community about healthy food. Your goal is to approve **all images that show healthy and nutritious food**. We encourage you to have a slightly higher standard as there is limited space to showcase these images in your community.

Here are a few in-scope examples,

<img src="https://cdn.prod.website-files.com/63bf3f41c3bc69578d823f01/6696f09f8741e2a1e4afeb05_plate%20of%20food.jpg" alt="Salon Hair Coloring Service" title="Hair Coloring Example" height="300"/>
<img src="https://healthylittlevittles.com/wp-content/uploads/2022/02/Anti-Inflammatory-Vegan-Green-Smoothie-4.jpg" alt="Salon Hair Coloring Service" title="Hair Coloring Example" height="300"/>

and a few out-of-scope examples.

The first image is out of scope because it contains fries, while the second image depicts smoothies with heavy cream.

<img src="https://www.mccain.co.uk/media/y4aitvlg/crispy-salmon-zesty-home-chips.jpg" alt="Salon Hair Coloring Service" title="Hair Coloring Example" height="300"/>
<img src="https://www.centercutcook.com/wp-content/uploads/2024/05/cookies-and-cream-smoothie.jpg" alt="Salon Hair Coloring Service" title="Hair Coloring Example" height="300"/>
<img src="https://natashaskitchen.com/wp-content/uploads/2024/03/Caesar-Salad-Dressing-copy.jpg" alt="Salon Hair Coloring Service" title="Hair Coloring Example" height="300"/>
<img src="https://www.flyinggoosebrand.com/wp-content/uploads/2022/09/Sriracha-mayo-sauce-can-be-enjoyed-on-nearly-any-food-including-sushi.-1.jpg" alt="Salon Hair Coloring Service" title="Hair Coloring Example" height="300"/>


In this notebook, you will help refine the definition of this concept, with the flexibility to address ambiguous scenarios that the initial description might not fully capture. Your definition will then be used as prompts for a language model to determine whether an image is in-scope or out-of-scope for this concept.

In [ ]:
#@markdown ## **Start with a simple description**
#@markdown We will start with a simple description of this concept.

#@markdown **Do not run this cell twice**
experiment_folder = "./outputs" # @param {type: "string"}
concept = 'Healthy Food' #@param {type:"string"}

slug_concept = slugify.slugify(concept)
experiment_name = 'concept_deliberation'
output_dir = f"{experiment_folder}/{slug_concept}/{experiment_name}"
os.makedirs(output_dir, exist_ok=True)

# Initialize the RetrievalClient with the indices_train.json file
from agile_deliberation_lib.retrieval import RetrievalClient
retrieval_client = RetrievalClient(indices_path="indices_train.json")

# Initialize ConceptDeliberator
concept_deliberator = deliberators_py.ConceptDeliberator(
    retrieval_client, model_client, definition_folder=output_dir, keep_output=False)
display_with_styles = components_py.display_with_styles

# @markdown Initial concept defiinition:
concept_description = "Images that show healthy food."  # @param {type:"string"}
definition = Definition(concept, concept_description)

In [ ]:
#@markdown ## **Step 1: Enrich Your Concept Definitions**
#@markdown Our system will begin by enriching the concept definition you've provided.

#@markdown You are asked to determine whether each signal is in-scope or out-of-scope.
#@markdown Reviewing relevant images can help you get a better idea of what this signal refers to.

concept_deliberator.enrich_definitions(definition)

In [ ]:
#@markdown ## **Step 2: Run this cell to review your resultant definition**

#@markdown As you create a sketch of your concept definition at the first stage,
#@markdown you are invited to spend two minutes reviewing your current definition.

#@markdown In particular, some signals might be repetitive, while others might no longer make sense to you.
#@markdown You could also adjust the language of each signal to make it more accurate here.

#@markdown If you want to add a new signal, also feel free to add it by clicking the button "Add New Signal".
display_with_styles(components_py.interactive_definition(definition))

In [ ]:
#@markdown ## **Step 3: Reflect on Borderline Images**
#@markdown Now that you have a clear outline of your concept definition, it's important to ensure that the language is free of ambiguities.
#@markdown We will present five images in each round for you to review and label.

#@markdown **If the classifier makes a different decision from you, you are encouraged to communicate your thoughts in more detail.**

#@markdown **Otherwise, you could also write your thoughts for images that you find reflecting important ambiguities.**

#@markdown Our system will then try its best to edit the definition according to your feedback.

#@markdown Finally, ***be patient***, as the classifier is still learning your decision-making criteria.

reflect_images_path = f'{experiment_folder}/{slug_concept}/datasets/cached_images_for_iterations.pkl'
log_path=None

concept_deliberator.image_reflection(definition, reflect_images_path=reflect_images_path, log_path=log_path)

In [ ]:
#@markdown ## Final Step: Annotate Groundtruth
#@markdown Then run this cell to annotate the following dataset.
#@markdown so that we could evaluate the performance of your definition.

#@markdown If you realize that your labels are inconsistent, feel free to go back to previous images to modify your annotations.
#@markdown You could use the 'left-arrow' and 'right arrow' keys to switch between images.

import random

dataset_filename = f'{experiment_folder}/{slug_concept}/datasets/test_dataset_for_evaluation.pkl'

# NOTE: This cell requires the `test_dataset_for_evaluation.pkl` file to exist.
# If you are running in Colab from scratch, you might need to upload this file or adjust the path.
with open(dataset_filename, 'rb') as f:
  test_images = pickle.load(f)
print(f'There are in total {len(test_images)} images for annotations')
test_images = random.sample(test_images, len(test_images))

test_annotations_path= f'{output_dir}/test_annotations_cache.pkl'
annotator = annotator_py.Annotator(
    test_images,
    mode='complete',
    cache_file_name=test_annotations_path,
)
annotations = annotator.annotate()